这是个零样本学习算法，因为我们没有提供任何提示

In [1]:
# 需先安装: pip install langchain-community
# 注意：load_tools 是“模块”也是“函数”，要从模块里显式导入函数
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",  # 本地服务可不校验，任意非空即可
    temperature=0,
    model="local",        # 若 LM Studio 要求指定模型名，可改为你在 LM Studio 里加载的模型名
)

tools= load_tools(["llm-math"], llm = llm)

agent= initialize_agent(
    tools,
    llm,
    agent= AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors = True,
    verbose= True)


/Users/ludan/project/chanon-data-lab/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_32347/1984436621.py:17: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-ag

In [2]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

In [3]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
# 纯问答用 llm 直接调用，避免 Agent 的 ReAct 格式解析失败
response = llm.invoke(prompt)
print(response.content)

positive


In [4]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
# 纯问答用 llm 直接调用，避免 Agent 的 ReAct 格式解析失败
response = llm.invoke(prompt)
print(response.content)

satisfied, grateful, positive, efficient, reliable


In [5]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
# 纯问答用 llm 直接调用，避免模型返回 JSON 导致 Agent 解析失败
response = llm.invoke(prompt)
print(response.content)

{
  "Item": "lamp",
  "Brand": "Lumina"
}


In [ ]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
# 纯问答用 llm 直接调用，避免模型返回 JSON 导致 Agent 解析失败
response = llm.invoke(prompt)
print(response.content)

{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "lamp",
  "Brand": "Lumina"
}


: 